In [ ]:
import sys; sys.path.append('..')
import MeshFEM, mesh, sparse_matrices, benchmark, field_sampler, mesh_utilities, parallelism
import inflatables_parametrization as parametrization, numpy as np, importlib, pickle, wall_generation, visualization
import utils
import py_newton_optimizer
from py_newton_optimizer import NewtonOptimizerOptions
from io_redirection import suppress_stdout
from py_newton_optimizer import NewtonOptimizerOptions
import wall_width_formulas as wwf
import numpy as np
from matplotlib import pyplot as plt
from tri_mesh_viewer import TriMeshViewer

In [ ]:
sphere = mesh.Mesh('../../examples/full_sphere.msh')

In [ ]:
view = TriMeshViewer(sphere)
view.showWireframe()
view.show()

In [ ]:
threshold = 0.8
bb = utils.bbox(sphere.vertices())
zThreshold = bb[1][2] * threshold + (1 - threshold) * bb[0][2]

# Delete the vertices below the Z threshold and then construct the triangles
# induced by this vertex subset.
keepVtx = sphere.vertices()[:, 2] >= zThreshold
V = sphere.vertices()[keepVtx, :]

vtxRenumber = -1 * np.ones(sphere.numVertices())
vtxRenumber[np.arange(sphere.numVertices())[keepVtx]] = np.arange(V.shape[0])

F = vtxRenumber[sphere.triangles()]
F = F[np.min(F, axis=1) >= 0]
target_surf = mesh.Mesh(V, F)

view.update(preserveExisting=False, mesh=target_surf)

In [ ]:
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=True))

In [ ]:
# Choose reasonable stretching bounds
alphaMin = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(2, 10))
alphaMax = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))
print(alphaMin, alphaMax)

In [ ]:
lg = parametrization.LocalGlobalParametrizer(target_surf, parametrization.lscm(target_surf))

for i in range(1000): lg.runIteration()
print(lg.energy())
lg.alphaMin = alphaMin
lg.alphaMax = alphaMax

print(lg.energy())
for i in range(1000): lg.runIteration()
print(lg.energy())

In [ ]:
print(lg.energy())
for i in range(8000): lg.runIteration()
print(lg.energy())

In [ ]:
rparam = parametrization.RegularizedParametrizerSVD(target_surf, lg.uv())
rparam.alphaMin = alphaMin
rparam.alphaMax = alphaMax

In [ ]:
def optimize_rparam(param, alphaRegW, phiRegW, bendRegW):
    param.alphaRegW = alphaRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegW
    opts = NewtonOptimizerOptions()
    opts.niter = 1000
    opts.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
    #opts.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()
    cr = parametrization.regularized_parametrization_newton(param, param.rigidMotionPinVars, opts)

In [ ]:
# Rerun this cell until convergence (twice)
with suppress_stdout(): optimize_rparam(rparam, 1e2, 1e0, 50.0)
with suppress_stdout(): optimize_rparam(rparam, 1e1, 1e-1, 50.0)

with suppress_stdout(): optimize_rparam(rparam, 1e0, 1e-2, 250.0)
with suppress_stdout(): optimize_rparam(rparam, 1e-1, 5e-3, 100.0)
with suppress_stdout(): optimize_rparam(rparam, 1e-1, 5e-3, 25.0)
with suppress_stdout(): optimize_rparam(rparam, 1e-2, 2.5e-3, 12.5)

In [ ]:
visualization.singularValueHistogram(rparam)

In [ ]:
visualization.visualize(rparam)

## Upsampling and channel generation

In [ ]:
nsubdiv=2
upsampledMesh, upsampledAngles, upsampledStretches = rparam.upsampledVertexLeftStretchAnglesAndMagnitudes(nsubdiv)
upsampledStretches = np.clip(upsampledStretches, alphaMin, alphaMax)
(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_stripe_field(upsampledMesh.vertices(), upsampledMesh.triangles(), upsampledAngles,
                                                                    wwf.canonicalWallWidthForStretchFactor(upsampledStretches), frequency=0.2)

In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, height=12)

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=4.0,
                                              minContourLen=10)

In [ ]:
visualization.plot_line_segments(pts, edges, width=20, height=16)

## Meshing and inflation simulation

In [ ]:
triArea = 6.0

In [ ]:
m, fuseMarkers, edgeMarkers = wall_generation.triangulate_channel_walls(pts[:,0:2], edges, triArea=triArea)
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=20, height=18)

In [ ]:
import sheet_meshing
m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, pts, edges, triArea=triArea)

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=0.5 / 20,
                                              minContourLen=0.75 / 20)

In [ ]:
visualization.plot_line_segments(pts, edges, width=10, height=8)

## Meshing and inflation simulation

In [ ]:
import parametrization, sparse_matrices, mesh, numpy as np, importlib, pickle, wall_generation
from tri_mesh_viewer import TriMeshViewer
import vis, matplotlib
from py_newton_optimizer import NewtonOptimizerOptions
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization, wall_width_formulas as wwf

In [ ]:
from sheet_meshing import generateSheetMesh
s, iwv, iwbv = generateSheetMesh(sdfVertices, sdfTris, sdf, triArea=0.05 / (20**2), permitWallInteriorVertices=False, targetEdgeSpacing=0.5 / 20, minContourLen=0.75 / 20)

In [ ]:
import inflation
isheet = inflation.InflatableSheet(s, iwv)
bv = isheet.mesh().boundaryVertices()
bdryVars = [isheet.varIdx(0, i, c) for i in bv for c in range(3)]

uv = rparam.uv()
paramSampler = mesh_utilities.SurfaceSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surface.triangles())
liftedSheetPositions = paramSampler.sample(s.vertices(), target_surface.vertices())

In [ ]:
isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
harmonicPositions = parametrization.harmonic(isheet.mesh(), liftedSheetPositions[bv])
harmonicPositions += 0.15 * (liftedSheetPositions - harmonicPositions)

In [ ]:
isheet.setUninflatedDeformation(harmonicPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
import tri_mesh_viewer
importlib.reload(tri_mesh_viewer)
viewer = tri_mesh_viewer.TriMeshViewer(isheet.visualizationMesh(), width=1024, height=640)
viewer.showWireframe()
viewer.show()

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 20
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-6
opts.niter = iterations_per_output

In [ ]:
targetAttractedSheet = inflation.TargetAttractedInflation(isheet, target_surface)
targetAttractedSheet.energy(targetAttractedSheet.EnergyType.Fitting)

In [ ]:
targetAttractedSheet.targetSurfaceFitter().holdClosestPointsFixed = False

In [ ]:
targetAttractedSheet.fittingWeight = 1e-8

In [ ]:
import time
inflation.benchmark_reset()
niter = 5000
iterations_per_output = 10
opts.niter = iterations_per_output
isheet.pressure = 0.25
fixedVars = bdryVars
#fixedVars = isheet.rigidMotionPinVars
for step in range(int(niter / iterations_per_output)):
    cr = inflation.inflation_newton(isheet, fixedVars, opts)
    viewer.update(False, isheet.visualizationMesh())
    # isheet.visualizationMesh().save(f'orig_design_inflation/step_{step}.msh')
    time.sleep(0.01) # Allow some mesh synchronization time for pythreejs
    if cr.numIters() < iterations_per_output: break
inflation.benchmark_report()